# Phase 1

In [3]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Classes :", data.target_names)

print("\nRépartition de la variable cible :")
display(y.value_counts(normalize=True).sort_index().round(3))

Classes : ['malignant' 'benign']

Répartition de la variable cible :


target
0    0.373
1    0.627
Name: proportion, dtype: float64

### découpage X et y en trois jeux 

In [4]:
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    

    if test_size <= 0 or test_size >= 1:
        raise ValueError("test_size doit être strictement compris entre 0 et 1.")

    if val_size <= 0 or val_size >= 1:
        raise ValueError("val_size doit être strictement compris entre 0 et 1.")

    if len(X) != len(y):
        raise ValueError("X et y doivent avoir le même nombre de lignes.")

    # Premier split : on isole le jeu de test
    X_reste, X_test, y_reste, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    # Deuxième split : on isole la validation dans le reste
    X_train, X_val, y_train, y_val = train_test_split(
        X_reste,
        y_reste,
        test_size=val_size,
        random_state=random_state,
        stratify=y_reste
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X,
    y,
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

print("Train :", len(X_train), "| Validation :", len(X_val), "| Test :", len(X_test))

Train : 364 | Validation : 91 | Test : 114


### Cas normal

In [5]:
total_initial = len(X)
total_apres_split = len(X_train) + len(X_val) + len(X_test)

print("Total initial :", total_initial)
print("Total après découpage :", total_apres_split)

if total_initial == total_apres_split:
    print("Checkpoint cas normal : les trois tailles s'additionnent correctement.")
else:
    print("Erreur : les tailles ne s'additionnent pas correctement.")

Total initial : 569
Total après découpage : 569
Checkpoint cas normal : les trois tailles s'additionnent correctement.


#### vérification  la répartition des classes

In [6]:
repartition_classes = pd.DataFrame({
    "dataset complet": y.value_counts(normalize=True).sort_index(),
    "train": y_train.value_counts(normalize=True).sort_index(),
    "validation": y_val.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index()
})

repartition_classes = repartition_classes.round(3)

print("Répartition des classes dans chaque jeu :")
display(repartition_classes)

Répartition des classes dans chaque jeu :


,dataset complet,train,validation,test
target,,,,
0,0.373,0.374,0.374,0.368
1,0.627,0.626,0.626,0.632


#### conclusion automatique sur la stratification

In [7]:
ecart_max = (
    repartition_classes
    .sub(repartition_classes["dataset complet"], axis=0)
    .abs()
    .max()
    .max()
)

print("Écart maximal observé :", ecart_max)

if ecart_max <= 0.02:
    print("Répartition des classes conservée dans chaque jeu : oui")
else:
    print("Répartition des classes conservée dans chaque jeu : non")

Écart maximal observé : 0.0050000000000000044
Répartition des classes conservée dans chaque jeu : oui


### cas limite avec val_size=0

#### tester val_size=0

In [8]:
try:
    split_train_val_test(
        X,
        y,
        test_size=0.2,
        val_size=0,
        random_state=42
    )
except ValueError as erreur:
    print("Checkpoint cas limite : la fonction plante proprement.")
    print("Message d'erreur :", erreur)

Checkpoint cas limite : la fonction plante proprement.
Message d'erreur : val_size doit être strictement compris entre 0 et 1.


### Cas adversarial : dataset déséquilibré 95 % / 5 %

#### créer un dataset artificiel déséquilibré

In [11]:
rng = np.random.default_rng(42)

X_adversarial = pd.DataFrame(
    rng.normal(size=(1000, 5)),
    columns=["feature_1", "feature_2", "feature_3", "feature_4", "feature_5"]
)

y_adversarial = pd.Series(
    [0] * 950 + [1] * 50,
    name="target"
)

print("Taille du dataset adversarial :", len(X_adversarial))

print("\nRépartition initiale des classes :")
display(y_adversarial.value_counts(normalize=True).sort_index().round(3))

Taille du dataset adversarial : 1000

Répartition initiale des classes :


target
0    0.95
1    0.05
Name: proportion, dtype: float64

#### fonction au dataset déséquilibré

In [12]:
X_train_adv, X_val_adv, X_test_adv, y_train_adv, y_val_adv, y_test_adv = split_train_val_test(
    X_adversarial,
    y_adversarial,
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

print("Train :", len(X_train_adv), "| Validation :", len(X_val_adv), "| Test :", len(X_test_adv))

Train : 640 | Validation : 160 | Test : 200


#### vérifier la répartition 95/5 dans les trois jeux

In [13]:
repartition_adversarial = pd.DataFrame({
    "dataset complet": y_adversarial.value_counts(normalize=True).sort_index(),
    "train": y_train_adv.value_counts(normalize=True).sort_index(),
    "validation": y_val_adv.value_counts(normalize=True).sort_index(),
    "test": y_test_adv.value_counts(normalize=True).sort_index()
})

repartition_adversarial = repartition_adversarial.round(3)

print("Répartition des classes sur le dataset adversarial :")
display(repartition_adversarial)

Répartition des classes sur le dataset adversarial :


,dataset complet,train,validation,test
target,,,,
0,0.95,0.95,0.95,0.95
1,0.05,0.05,0.05,0.05


#### conclusion sur le cas adversarial

In [14]:
ecart_max_adversarial = (
    repartition_adversarial
    .sub(repartition_adversarial["dataset complet"], axis=0)
    .abs()
    .max()
    .max()
)

print("Écart maximal observé :", ecart_max_adversarial)

if ecart_max_adversarial <= 0.02:
    print("Checkpoint cas adversarial : stratify garde bien la répartition 95/5.")
else:
    print("Checkpoint cas adversarial : problème dans la conservation des proportions.")

Écart maximal observé : 0.0
Checkpoint cas adversarial : stratify garde bien la répartition 95/5.
